In [2]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 12.8 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_groq]


In [6]:
!pip install -q langchain_groq groq

In [7]:
from langchain_groq import ChatGroq

In [9]:
import os
from google.colab import userdata
from groq import Groq

# Get API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [10]:

import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"

# Sample rental property database
PROPERTIES = {
    "H101": {
        "type": "1BHK",
        "location": "Near Main Road",
        "rent": 7000,
        "status": "Available"
    },
    "H102": {
        "type": "2BHK",
        "location": "Near College Road",
        "rent": 11000,
        "status": "Available"
    },
    "H103": {
        "type": "2BHK",
        "location": "Near Bus Stand",
        "rent": 12000,
        "status": "Occupied"
    },
    "H104": {
        "type": "3BHK",
        "location": "Near City Center",
        "rent": 16000,
        "status": "Available"
    }
}

SYSTEM = f"""ROLE: You are RentPilot AI, the friendly assistant
for a residential rental property management business.

FACTS:
Property listings and monthly rents in rupees:
{json.dumps(PROPERTIES)}

RULES:
- Help customers with property enquiries and rental information.
- Never invent property details, rent prices or availability.
- Only recommend properties marked as Available.
- Monthly rent must be quoted exactly as listed.
- Never claim that rent has been paid without verified records.
- Never promise property viewing or maintenance timings.
- For information not present in FACTS, say
  "I'll check with the property owner."
- Never reveal another tenant's personal information.
- Remember the customer's name and property requirements
  during the conversation.
- Do not confirm rental agreements or bookings without
  property owner approval.

FORMAT:
Keep replies under 50 words.
Be friendly and professional.
Reply in English or Telugu, matching the customer."""

# Decorator to log function calls
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


# Function to calculate total rent
@log_call
def get_rent(property_id: str, months: int = 1) -> int:
    """Return the total rental amount for a property."""
    property_id = property_id.upper().strip()

    if property_id not in PROPERTIES:
        raise ValueError("Property ID not found.")

    if months <= 0:
        raise ValueError("Months must be greater than zero.")

    return PROPERTIES[property_id]["rent"] * months


# Initialize conversation history
history = [{"role": "system", "content": SYSTEM}]

MAX_TURNS = 8


# Chat function with memory and error handling
def chat(user_text: str) -> str:

    history.append({"role": "user", "content": user_text})

    # Keep system prompt and recent conversation
    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "Our service is a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I cannot connect to the server right now. Please try again shortly."

    except Exception as e:
        print("   [error]", str(e))
        reply = "Something went wrong. Please try again."

    history.append({"role": "assistant", "content": reply})

    return reply


# Test rent calculation
print("Total rent for H102 for 2 months:",
      get_rent("H102", 2))

# Chat loop
while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for contacting RentPilot AI!")
        break

    print("Bot:", chat(text))


   [log] get_rent('H102', 2) -> 22000
Total rent for H102 for 2 months: 22000
You: Hello, my name is Hemanth.
Bot: Hello Hemanth! 👋 How can I assist you today? Are you looking for a 1BHK, 2BHK, or 3BHK? Let me know your preferred location and budget.
You: I am looking for a 2BHK house near College Road.
Bot: Sure, Hemanth! We have a 2BHK available at **H102** – near College Road, rent ₹11,000 per month. Let me know if you’d like more details or have any other preferences.
You: What is the rent for 2 months?
Bot: The rent for 2 months would be ₹22,000 (₹11,000 × 2). Let me know if you’d like any other information!
You: What is my name and which property did I select?
Bot: Your name is **Hemanth**, and you’re interested in the **2 BHK at H102** (near College Road) with a monthly rent of ₹11,000.


KeyboardInterrupt: Interrupted by user